# Week 4: Threshold Sensitivity & Zero-Day Recall
**Risk-Adaptive Zero Trust — Robustness Analysis**

**Author:** Okado Aaron Obbo — S24B23/015
**Institution:** Uganda Christian University

---

This notebook documents the Week 4 robustness analysis: how the autoencoder anomaly detector performs across different detection thresholds, and whether it can detect attack types it has never encountered during threshold calibration (zero-day scenario).

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('Setup complete')

## 1. Introduction

The autoencoder is trained on NORMAL traffic only using self-supervised learning. It flags anomalies when reconstruction error exceeds a threshold computed as the 95th percentile of reconstruction errors on normal test traffic.

Week 4 investigates two questions:
1. **Threshold Sensitivity**: How do precision, recall, F1, and FPR change as the threshold is varied from the 70th to the 99th percentile?
2. **Zero-Day Recall**: If an entire attack class (DoS, Probe, R2L, or U2R) had never been seen before, would the autoencoder still detect it using only the threshold calibrated on normal traffic?

## 2. Threshold Sensitivity Analysis

In [ ]:
pd.set_option('display.max_columns', None)
df = pd.read_csv('c:/Users/AARON/Desktop/Final Year Project Prototype/zta-ml-framework/data/results/threshold_sensitivity.csv')
display(df)

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename='c:/Users/AARON/Desktop/Final Year Project Prototype/zta-ml-framework/data/results/threshold_sensitivity.png'))

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename='c:/Users/AARON/Desktop/Final Year Project Prototype/zta-ml-framework/data/results/threshold_roc_curve.png'))

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename='c:/Users/AARON/Desktop/Final Year Project Prototype/zta-ml-framework/data/results/threshold_pr_curve.png'))

In [ ]:
with open('c:/Users/AARON/Desktop/Final Year Project Prototype/zta-ml-framework/data/results/robustness_report.json') as f:
    report = json.load(f)
ts = report['threshold_sensitivity']
print(f"Optimal F1 percentile : {ts['optimal_f1_percentile']}")
print(f"Optimal F1 threshold  : {ts['optimal_f1_threshold']}")
print(f"Optimal F1 metrics    : {ts['optimal_f1_metrics']}")

## 3. Zero-Day Recall Analysis

For each attack class, we hold it out as "unseen" and measure what fraction the autoencoder detects using only the 95th-percentile threshold calibrated on normal traffic. This directly simulates zero-day detection capability.

In [ ]:
df_zd = pd.read_csv('c:/Users/AARON/Desktop/Final Year Project Prototype/zta-ml-framework/data/results/zeroday_recall.csv')
display(df_zd)

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename='c:/Users/AARON/Desktop/Final Year Project Prototype/zta-ml-framework/data/results/zeroday_recall.png'))

In [ ]:
with open('c:/Users/AARON/Desktop/Final Year Project Prototype/zta-ml-framework/data/results/robustness_report.json') as f:
    report = json.load(f)
zd = report['zeroday_recall']
print(f"Hardest class to detect (lowest zero-day recall): {zd['hardest_class']}")
print(f"Easiest class to detect (highest zero-day recall): {zd['easiest_class']}")

## 4. Robustness Summary

In [ ]:
with open('c:/Users/AARON/Desktop/Final Year Project Prototype/zta-ml-framework/data/results/robustness_report.json') as f:
    report = json.load(f)
print(json.dumps(report, indent=2))

## 5. Conclusions

- The threshold sensitivity sweep shows the trade-off between precision and recall: lower percentiles catch more attacks (higher recall) but increase false positives; higher percentiles reduce false alarms but may miss more attacks.
- Zero-day recall is highest for DoS and Probe attack classes, which produce reconstruction errors that clearly exceed the normal-traffic threshold — even without ever appearing in threshold calibration.
- R2L has the lowest zero-day recall because R2L connections (e.g., password guessing, FTP write) share many features with normal user sessions and reconstruct with low error.
- The 95th-percentile threshold is a reasonable operating point, but the optimal-F1 threshold from the sweep may differ and provides a data-driven alternative.